# Embedding + ChromaDB Pipeline - AutoDrive RAG

Ce notebook réalise :
- Lecture des chunks JSONL produits par le preprocessing
- Génération des embeddings avec BAAI/bge-m3
- Création d'une base ChromaDB persistante
- Création d'une collection par agent métier
- Test de recherche vectorielle

Collections créées : assurances, contrats, faq, maintenance, vehicules.

In [4]:
# Installation (à exécuter une seule fois)
pip install sentence-transformers chromadb tqdm


SyntaxError: invalid syntax (2530137738.py, line 2)

In [3]:
import json
from pathlib import Path
from tqdm import tqdm
import chromadb
from sentence_transformers import SentenceTransformer


ModuleNotFoundError: No module named 'tqdm'

## 1. Chargement du modèle embedding

BGE-M3 est choisi car il supporte le multilingue et convient aux recherches métier français/arabe.

In [ ]:
model = SentenceTransformer('BAAI/bge-m3')
print('Model loaded')


In [ ]:
# Modifier ce chemin selon ton dossier
DATA_DIR = Path("/home/tawil/Bureau/stage d'ete /preprocessing/output/")
CHROMA_DIR = Path("/home/tawil/Bureau/stage d'ete/preprocessing/output/")
FILES = {
    'assurances':'chunks_assurances.jsonl',
    'contrats':'chunks_contrats.jsonl',
    'faq':'chunks_faq.jsonl',
    'maintenance':'chunks_maintenance.jsonl',
    'vehicules':'chunks_vehicules.jsonl'
}


In [ ]:
def load_jsonl(path):
    data=[]
    with open(path,'r',encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Exemple vérification
sample = load_jsonl(DATA_DIR / FILES['assurances'])
print('Chunks assurances:', len(sample))
print(sample[0].keys())


## 2. Création ChromaDB

In [ ]:
client = chromadb.PersistentClient(path=CHROMA_DIR)
print('ChromaDB ready')


In [ ]:
def build_collection(name, filename, batch_size=64):
    print('\nProcessing:', name)
    chunks = load_jsonl(DATA_DIR / filename)

    collection = client.get_or_create_collection(name=name)

    for i in tqdm(range(0,len(chunks),batch_size)):
        batch = chunks[i:i+batch_size]

        texts=[x['text'] for x in batch]
        ids=[x.get('chunk_id',f'{name}_{i+j}') for j,x in enumerate(batch)]

        embeddings=model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()

        metadatas=[]
        for x in batch:
            meta=x.get('metadata',{})
            meta['chunk_type']=x.get('chunk_type','')
            meta['chunk_index']=str(x.get('chunk_index',''))
            metadatas.append(meta)

        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas
        )

    print('Stored:', collection.count())


for agent,file in FILES.items():
    build_collection(agent,file)


## 3. Test Retrieval


In [ ]:
collection = client.get_collection('assurances')

question='Quelle est la franchise et les exclusions CDW ?'

q_embedding=model.encode(
    question,
    normalize_embeddings=True
).tolist()

result=collection.query(
    query_embeddings=[q_embedding],
    n_results=3
)

for doc,meta in zip(result['documents'][0],result['metadatas'][0]):
    print('---')
    print(doc[:500])
    print(meta)
